we will use gradio for demo

In [1]:
!pip install -q gradio
!pip install -q faiss-cpu sentence-transformers
!pip install -q pandas chromadb sentence-transformers transformers accelerate bitsandbytes
!pip install -q bitsandbytes>=0.46.1

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
import os
import json
import pandas as pd
import numpy as np
import faiss
import gradio as gr
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# =========================
# PATHS
# =========================
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/intelligence_ai_marketing"
ADS_CSV = f"{BASE_DIR}/data/ads_with_strategies.csv"
INSIGHT_DIR = f"{BASE_DIR}/analytics/insights"

# =========================
# LOAD DATA
# =========================
ads = pd.read_csv(ADS_CSV)

with open(f"{INSIGHT_DIR}/strategy_insights.json", "r") as f:
    strategy_insights = json.load(f)

with open(f"{INSIGHT_DIR}/brand_strategy_patterns.json", "r") as f:
    brand_patterns = json.load(f)

with open(f"{INSIGHT_DIR}/visual_design_patterns.json", "r") as f:
    visual_patterns = json.load(f)

# =========================
# BUILD DOCUMENTS
# =========================
def row_to_document(row):
    parts = []

    industry = row.get("competitor", "")
    if pd.notna(industry) and str(industry).strip():
        parts.append(f"Industry: {industry}")

    category = row.get("all_categories_full", "")
    if pd.notna(category) and str(category).strip():
        parts.append(f"Category: {category}")

    strategy = row.get("strategy", "")
    if pd.notna(strategy) and str(strategy).strip():
        parts.append(f"Strategy: {strategy}")

    strategy_group = row.get("strategy_group", "")
    if pd.notna(strategy_group) and str(strategy_group).strip():
        parts.append(f"Strategy group: {strategy_group}")

    sentiment = row.get("sentiment_polarity", "")
    if pd.notna(sentiment):
        parts.append(f"Sentiment score: {sentiment}")

    keywords = row.get("top_keywords", "")
    if pd.notna(keywords) and str(keywords).strip():
        parts.append(f"Keywords: {keywords}")

    ocr = row.get("ocr_text", "")
    if pd.notna(ocr) and str(ocr).strip():
        parts.append(f"Ad text: {str(ocr)[:400]}")

    layout = row.get("layout_type", "")
    if pd.notna(layout) and str(layout).strip():
        parts.append(f"Layout type: {layout}")

    colors = []
    for i in range(1, 6):
        c = row.get(f"dominant_color_{i}", "")
        if pd.notna(c) and str(c).strip():
            colors.append(str(c))
    if colors:
        parts.append("Colors: " + ", ".join(colors))

    return " | ".join(parts)

documents = [row_to_document(ads.iloc[i]) for i in range(len(ads))]

# =========================
# VECTOR SEARCH
# =========================
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(documents, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

def retrieve_docs(question, top_k=2):
    q_embedding = embed_model.encode([question]).astype("float32")
    distances, indices = index.search(q_embedding, top_k)
    return [documents[i] for i in indices[0]]

# =========================
# LOAD QWEN
# =========================
model_name = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# =========================
# CLEANER
# =========================
def clean_consulting_output(text):
    section_names = [
        "MARKET PATTERN",
        "STRATEGIC INSIGHT",
        "COMPETITIVE GAP",
        "RECOMMENDATION",
        "CURRENT PATTERN",
        "GAP",
        "AD CONCEPT",
        "HEADLINE",
        "CORE MESSAGE",
        "VISUAL CONCEPT",
        "CTA"
    ]

    lines = text.splitlines()
    cleaned_lines = []
    seen_sections = set()

    for line in lines:
        stripped = line.strip()
        normalized = stripped.replace("### ", "").replace("## ", "").strip()

        if normalized in section_names:
            if normalized in seen_sections:
                break
            seen_sections.add(normalized)

        if stripped.startswith("Do not add"):
            continue
        if stripped.startswith("Stop after"):
            continue
        if stripped.startswith("Keep the answer"):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines).strip()

# =========================
# CORE RAG
# =========================
def run_rag(question: str, task_prompt: str, top_k: int = 2, max_new_tokens: int = 180, temperature: float = 0.3):
    torch.cuda.empty_cache()

    docs = retrieve_docs(question, top_k=top_k)
    docs = [d[:450] for d in docs]
    context = "\n\n---\n\n".join(docs)

    prompt = f"""
You are a senior marketing strategy consultant.

You have access to a structured advertising dataset.

==============================
DATASET INSIGHTS
==============================

Strategy patterns:
{strategy_insights}

Brand strategy patterns:
{brand_patterns}

Visual design patterns:
{visual_patterns}

==============================
RELEVANT ADS
==============================

{context}

==============================
USER QUESTION
==============================

{question}

==============================
TASK
==============================

{task_prompt}
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return clean_consulting_output(response.strip())

# =========================
# FEATURES
# =========================
def analyze_industry_strategy(industry):
    question = f"What marketing strategies are commonly used in the {industry} industry?"

    task_prompt = """
Answer like a marketing strategy consultant.

Use this structure exactly once:

MARKET PATTERN
Describe the main pattern in the dataset.

STRATEGIC INSIGHT
Explain what this means for the industry.

COMPETITIVE GAP
Identify what competitors may be underusing or missing.

RECOMMENDATION
Provide a practical recommendation.

Do not repeat any section.
Do not add hashtags.
"""

    return run_rag(question, task_prompt, top_k=2, max_new_tokens=180, temperature=0.3)

def find_strategy_gap(industry):
    question = f"What opportunities are competitors in the {industry} industry missing, and how could a new brand differentiate?"

    task_prompt = """
Answer like a strategy consultant.

Use this structure exactly once:

CURRENT PATTERN
What are competitors mostly doing now?

GAP
What are they underusing or missing?

RECOMMENDATION
How should a new brand differentiate?

Do not repeat any section.
Do not add hashtags.
"""

    return run_rag(question, task_prompt, top_k=2, max_new_tokens=170, temperature=0.3)

def generate_ad_concept(industry, target_audience, style):
    question = f"""
Generate an advertising concept for a {industry} brand targeting {target_audience}.
Use a {style} style.
"""

    task_prompt = """
Generate a creative ad concept based on the dataset patterns.

Use this structure exactly once:

AD CONCEPT

HEADLINE
One short ad headline.

CORE MESSAGE
One or two sentences.

VISUAL CONCEPT
Describe the visual direction.

CTA
One short call-to-action.

Do not repeat sections.
Do not add hashtags.
"""

    return run_rag(question, task_prompt, top_k=2, max_new_tokens=180, temperature=0.6)

def get_reference_ads(industry):
    docs = retrieve_docs(f"{industry} advertising strategy", top_k=2)
    return "\n\n---\n\n".join(docs)

def full_demo(industry, audience, style):
    strategy = analyze_industry_strategy(industry)
    gap = find_strategy_gap(industry)
    concept = generate_ad_concept(industry, audience, style)
    refs = get_reference_ads(industry)
    return strategy, gap, concept, refs


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9706764cdc17097e9f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
# =========================
# UI
# =========================
import gradio as gr

def ui_analyze_strategy(industry, audience, style):
    return analyze_industry_strategy(industry)

def ui_find_gap(industry, audience, style):
    return find_strategy_gap(industry)

def ui_generate_concept(industry, audience, style):
    return generate_ad_concept(industry, audience, style)

def ui_show_refs(industry, audience, style):
    return get_reference_ads(industry)

with gr.Blocks() as demo:
    gr.Markdown("# AI Marketing Strategy Assistant")
    gr.Markdown(
        "Use the assistant to analyze industry strategy, identify competitor gaps, "
        "generate ad concepts, and retrieve reference ads from the dataset."
    )

    with gr.Row():
        industry = gr.Textbox(label="Industry", value="beauty")
        audience = gr.Textbox(label="Target Audience", value="Gen Z")
        style = gr.Textbox(label="Campaign Style", value="minimal and elegant")

    with gr.Row():
        btn_strategy = gr.Button("Analyze Strategy")
        btn_gap = gr.Button("Find Competitive Gap")
        btn_concept = gr.Button("Generate Ad Concept")
        btn_refs = gr.Button("Show Reference Ads")

    output = gr.Textbox(label="Assistant Output", lines=18)

    btn_strategy.click(
        fn=ui_analyze_strategy,
        inputs=[industry, audience, style],
        outputs=output
    )

    btn_gap.click(
        fn=ui_find_gap,
        inputs=[industry, audience, style],
        outputs=output
    )

    btn_concept.click(
        fn=ui_generate_concept,
        inputs=[industry, audience, style],
        outputs=output
    )

    btn_refs.click(
        fn=ui_show_refs,
        inputs=[industry, audience, style],
        outputs=output
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6c9c576d6c69726c90.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


save the file

In [7]:
app_code = r'''
PASTE THE FULL app.py CODE HERE
'''

with open("/content/drive/MyDrive/Colab Notebooks/intelligence_ai_marketing/ui/app.py", "w") as f:
    f.write(app_code)

print("app.py saved")

app.py saved


In [8]:
!python "/content/drive/MyDrive/Colab Notebooks/intelligence_ai_marketing/ui/app.py"

  File "/content/drive/MyDrive/Colab Notebooks/intelligence_ai_marketing/ui/app.py", line 2
    PASTE THE FULL app.py CODE HERE
          ^^^
SyntaxError: invalid syntax
